
# Multi-Agent Supervisor — Databricks

## Project Overview

This notebook demonstrates an agentic multi-agent architecture using Databricks.

We will create a Supervisor Agent that coordinates two existing specialist agents:

1. **External Tools Research Agent**
   - Performs external research.
   - Uses external tools to gather information.

2. **Customer Service Agent**
   - Retrieves internal claims information.
   - Looks up member policies and hospital details.
   - Uses Databricks AI Search for policy document retrieval.

The Supervisor Agent acts as the coordinator. It understands the user's objective, creates a plan, delegates tasks to specialist agents, evaluates their responses, and decides whether additional work is required.

### Main Objective

Demonstrate how multiple specialized agents can collaborate under an agentic supervisor to solve complex tasks.



## 1. System Architecture

The system consists of three agents.

### Supervisor Agent
- Understands the user's request.
- Breaks complex requests into smaller tasks.
- Selects the appropriate specialist agents.
- Evaluates their results.
- Requests additional work if information is missing.
- Combines the results into a final response.

### Research Agent
- Handles external research.
- Uses external tools to collect public information.
- Returns research findings to the supervisor.

### Customer Service Agent
- Handles internal customer service questions.
- Retrieves claims and policy information.
- Uses the existing claims database and AI Search tools.

### Workflow

User Request → Supervisor → Specialist Agents → Result Evaluation → Final Response

The supervisor may repeat delegation when the available results are insufficient.


![](/Workspace/Users/kiritjain@cyntexa.com/Presentation/Architecture.png)

![](/Workspace/Users/kiritjain@cyntexa.com/Presentation/Supervisor Workflow Process Diagram.webp)


## 2. Environment Setup

We will use the existing Databricks environment.

Required libraries:
- LangChain: to create and manage agents.
- LangGraph: to support agent orchestration.
- Databricks LangChain integration: to connect to Model Serving.
- MLflow: to track and evaluate the supervisor.

Since our two specialist agents already work, we should avoid unnecessarily upgrading their dependencies. We will install only missing packages if required.


In [0]:
%pip install --upgrade -q databricks-langchain ddgs wikipedia youtube-search langchain-community "mcp>=1,<2" "langchain==1.2.10" "langgraph==1.0.10" "langgraph-prebuilt==1.0.8"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



## 3. Import Required Libraries

We import the following components:

- `ChatDatabricks`: connects our supervisor to a Databricks Model Serving endpoint.
- `create_agent`: creates an agent that can decide when to call its tools.
- `tool`: converts Python functions into tools that an agent can use.
- `mlflow`: tracks our experiment and model execution.


In [0]:
import mlflow

from langchain_core.tools import tool
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks

print("All imports completed successfully.")

All imports completed successfully.



## 4. Initialize the Supervisor LLM

The supervisor needs its own language model to make decisions.

We will use the Databricks Model Serving endpoint:

`databricks-gpt-oss-20b`

This model will be responsible for:
- Understanding user requests.
- Deciding which specialist agent to call.
- Evaluating specialist responses.
- Deciding whether additional work is required.

The specialist agents can continue using their existing model endpoints.


In [0]:
SUPERVISOR_MODEL_ENDPOINT = "databricks-gpt-oss-20b"

supervisor_llm = ChatDatabricks(
    endpoint=SUPERVISOR_MODEL_ENDPOINT,
    temperature=0.1
)

print("Supervisor LLM initialized successfully.")


Supervisor LLM initialized successfully.



## 5. Create an MLflow Experiment

We will use MLflow to organize our supervisor experiments.

This will help us track:
- Different supervisor runs.
- Which specialist agents were called.
- How many delegation steps were performed.
- The final output of each run.

We will use a separate experiment so that supervisor runs do not get mixed with the individual agent experiments.


In [0]:

mlflow.set_experiment(
    "/Users/kiritjain@cyntexa.com/Multi_Agent_Supervisor"
)

print("MLflow experiment configured.")


MLflow experiment configured.


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:

# STEP 1: Check MLflow GenAI evaluation support

import mlflow

print("MLflow version:", mlflow.__version__)

# Check core GenAI evaluation APIs
try:
    from mlflow.genai import evaluate, scorer
    from mlflow.genai.scorers import (
        RelevanceToQuery,
        Guidelines
    )

    print("GenAI evaluation: AVAILABLE")
    print("RelevanceToQuery: AVAILABLE")
    print("Guidelines: AVAILABLE")

except Exception as e:
    print("GenAI evaluation import issue:", repr(e))


# Check custom LLM judge support
try:
    from mlflow.genai.judges import make_judge

    print("Custom make_judge(): AVAILABLE")

except Exception as e:
    print("Custom judge import issue:", repr(e))


# Enable LangChain tracing, if supported
try:
    mlflow.langchain.autolog()
    print("LangChain automatic tracing: ENABLED")

except Exception as e:
    print("LangChain autolog issue:", repr(e))


MLflow version: 3.18.0
GenAI evaluation: AVAILABLE
RelevanceToQuery: AVAILABLE
Guidelines: AVAILABLE
Custom make_judge(): AVAILABLE
LangChain automatic tracing: ENABLED


## 6. Connect the Existing Specialist Agents

We have already created two independent specialist agents and their lightweight setup notebooks.

Instead of copying all their initialization code into this notebook, we will use Databricks `%run` to load the setup notebooks.

We will connect:

1. External Tools Research Agent
2. Customer Service Agent

Each setup notebook initializes its specialist agent and the dependencies required by that agent.

### Important

The setup notebooks must assign their final agent objects to these exact variable names:

- `research_agent`
- `customer_service_agent`

The Research Agent setup must preserve its external research tools.

The Customer Service Agent setup must preserve its internal claims tools, AI Search policy-document tool, and verified-member authorization safeguards.

We will not modify or rebuild the original full agent notebooks.

In [0]:
%run "/Workspace/Users/kiritjain@cyntexa.com/Presentation/Research_Agent_Setup"

Research agent dependencies already present - skipping install.


In [0]:


# ---------------------------------------------------------
# RESEARCH AGENT CONNECTION
# ---------------------------------------------------------



print("Research Agent setup notebook executed.")

if "research_agent" not in globals():
    raise NameError(
        "Research Agent was not initialized by its setup notebook."
    )

print("Research Agent connected successfully.")

Research Agent setup notebook executed.
Research Agent connected successfully.


In [0]:
%run "/Workspace/Users/kiritjain@cyntexa.com/Presentation/Customer_Service_Agent_Setup"

Imports complete
Databricks user: kiritjain@cyntexa.com


In [0]:
# ---------------------------------------------------------
# CUSTOMER SERVICE AGENT CONNECTION
# ---------------------------------------------------------

# Replace this with the actual workspace path
# of your Customer Service Agent Setup notebook.


print("Customer Service Agent setup notebook executed.")

if "customer_service_agent" not in globals():
    raise NameError(
        "Customer Service Agent was not initialized by its setup notebook."
    )

print("Customer Service Agent connected successfully.")

Customer Service Agent setup notebook executed.
Customer Service Agent connected successfully.



## 7. Validate Specialist Agent Connections

Before building the supervisor, we must confirm that both specialist agents exist.

This is important because the supervisor will call these objects through Python functions.

If either agent is missing, we should fix the connection before proceeding.


In [0]:

if "research_agent" not in globals():
    raise NameError("Research Agent has not been initialized.")

if "customer_service_agent" not in globals():
    raise NameError("Customer Service Agent has not been initialized.")

if research_agent is None:
    raise ValueError("Research Agent is None.")

if customer_service_agent is None:
    raise ValueError("Customer Service Agent is None.")

print("Both specialist agents are connected successfully.")


Both specialist agents are connected successfully.



## 8. Create Specialist Agent Adapters

The supervisor should not need to understand how each specialist agent is implemented internally.

We will create two Python functions:

1. `run_research_agent()`
2. `run_customer_service_agent()`

Each function accepts a task, sends it to the corresponding specialist, and returns its response.

This creates a common interface between the supervisor and both specialist agents.

We will also create a helper function to extract readable text from the agent's response.


In [0]:

def extract_agent_response(result):

    # If the agent directly returns a string
    if isinstance(result, str):
        return result

    # Most LangChain create_agent calls return a dictionary
    if isinstance(result, dict):

        messages = result.get("messages", [])

        if messages:
            last_message = messages[-1]

            if hasattr(last_message, "content"):
                return str(last_message.content)

            if isinstance(last_message, dict):
                return str(last_message.get("content", last_message))

        # Check other common response keys
        for key in ["output", "answer", "response", "result"]:
            if key in result:
                return str(result[key])

    return str(result)


print("Response extraction function created.")


Response extraction function created.


In [0]:
def invoke_specialist(agent, task):

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": task
            }
        ]
    })

    return extract_agent_response(result)


def run_research_agent(task):

    print("\n[Supervisor -> Research Agent]")
    print("Task:", task)

    result = invoke_specialist(
        research_agent,
        task
    )

    print("[Research Agent completed]")

    return result


def run_customer_service_agent(task):

    print("\n[Supervisor -> Customer Service Agent]")
    print("Task:", task)

    result = invoke_specialist(
        customer_service_agent,
        task
    )

    print("[Customer Service Agent completed]")

    return result


print("Both specialist adapter functions are ready.")

Both specialist adapter functions are ready.


## 9. Test the Specialist Adapters

Before giving the supervisor access to the agents, we should test both adapters independently.

This verifies that:
- The supervisor can invoke the Research Agent.
- The supervisor can invoke the Customer Service Agent.
- Both agents return readable responses.

We are testing the connection here, not the supervisor's decision-making yet.


In [0]:

research_test = run_research_agent(
    "Explain what external research capabilities you have. "
    "Give a short summary of the tools you can use."
)

print("\nResearch Agent Response:")
print(research_test)



[Supervisor -> Research Agent]
Task: Explain what external research capabilities you have. Give a short summary of the tools you can use.
[Research Agent completed]

Research Agent Response:
I have access to several external research tools that allow me to gather information from various sources. These tools include:

1. Wikipedia search: This tool enables me to search for encyclopedic background information on people, places, companies, and historical events.
2. DuckDuckGo search: This tool allows me to search the web for current public information on a wide range of topics.
3. YouTube search: This tool enables me to search for videos on YouTube related to a specific topic or search query.

These tools can be useful for finding information, answering questions, and providing context on various subjects.


Trace(trace_id='tr-d537672ec1385e2bc74f125660546865')

In [0]:

customer_service_test = run_customer_service_agent(
    "Briefly explain what customer service information "
    "you can retrieve using your available tools."
)

print("\nCustomer Service Agent Response:")
print(customer_service_test)



[Supervisor -> Customer Service Agent]
Task: Briefly explain what customer service information you can retrieve using your available tools.
[Customer Service Agent completed]

Customer Service Agent Response:
I can help you with the following information:

| Tool | What it gives you |
|------|-------------------|
| **get_claim_details** | Claim status, dates, amounts, plan, and hospital for a specific claim ID. |
| **get_member_policies** | Current policies, plan benefit fields, and coverage details for the verified member (MEM‑0317). |
| **get_hospital_info** | Directory details (address, phone, services) for a hospital when you provide its internal ID. |
| **search_hospital_directory** | Finds hospital IDs, names, cities, and states when you give a hospital name. |

Let me know which of these you’d like to look up!


Trace(trace_id='tr-8ebb2c31d8b4cce6875f048ec9c49762')


## 10. Convert Specialist Agents into Tools

Now we will expose both specialist agents as tools.

Why?

The supervisor is itself an LLM-powered agent. It needs tools that allow it to delegate tasks to other agents.

We will create:

1. `delegate_to_research_agent`
2. `delegate_to_customer_service_agent`

The supervisor will decide when to call these tools.

For example:
- An external research question may require only the Research Agent.
- A claim status question may require only the Customer Service Agent.
- A hospital verification task may require both agents.

The supervisor will make this decision dynamically.


In [0]:
@tool
def delegate_to_research_agent(task: str) -> str:
    """
    Delegate external research and public information
    gathering to the External Tools Research Agent.
    """

    return run_research_agent(task)


@tool
def delegate_to_customer_service_agent(task: str) -> str:
    """
    Delegate internal claims, member policy, hospital
    directory, and policy document questions to the
    Customer Service Agent.
    """

    return run_customer_service_agent(task)


supervisor_tools = [
    delegate_to_research_agent,
    delegate_to_customer_service_agent
]

print("Supervisor tools created:")

for item in supervisor_tools:
    print("-", item.name)

Supervisor tools created:
- delegate_to_research_agent
- delegate_to_customer_service_agent



## 11. Define the Supervisor's Behaviour

This is one of the most important cells in the notebook.

We will give the supervisor instructions describing how it should behave.

It must not simply route every question to one agent.

Instead, it should:

1. Understand the user's objective.
2. Break complex objectives into subtasks.
3. Select the appropriate specialist agents.
4. Delegate tasks.
5. Evaluate their responses.
6. Identify missing or conflicting information.
7. Decide whether another specialist call is required.
8. Produce a final answer when sufficient information is available.

This decision-making cycle is what makes the supervisor agentic.

The model is not guaranteed to make perfect decisions, so we will test its behaviour later.


In [0]:

SUPERVISOR_SYSTEM_PROMPT = """
You are an intelligent Supervisor Agent coordinating
two specialist AI agents.

YOUR SPECIALISTS:

1. RESEARCH AGENT
   - Handles external research.
   - Gathers publicly available information.
   - Uses Wikipedia, web search, YouTube search,
     and other available external research tools.
   - Reports findings and any uncertainty.

2. CUSTOMER SERVICE AGENT
   - Handles internal claims information.
   - Retrieves member and policy information.
   - Searches the internal hospital directory.
   - Retrieves information from internal policy documents.
   - Must always preserve existing authorization
     and verified-member safeguards.

YOUR RESPONSIBILITIES:

1. Understand the user's complete objective.

2. Break complex requests into smaller, focused tasks.

3. Decide which specialist agents are required.

4. You may call one specialist or both specialists.

5. When a request requires both internal and external
   information, delegate to BOTH specialists.

6. Give each specialist a clear and focused task.

7. Carefully inspect every response returned by
   the specialist agents.

8. Evaluate whether the results are sufficient,
   relevant, and consistent.

9. Identify missing information, ambiguity,
   contradictions, or possible entity mismatches.

10. If additional information is required, decide which
    specialist should perform a focused follow-up task.

11. You may delegate follow-up tasks more than once,
    but every additional call must have a clear purpose.

12. Avoid repeating the same task when a previous
    specialist response has already answered it.

HOSPITAL DIRECTORY INVESTIGATION RULES:

13. When the user provides a hospital name, ask the
    Customer Service Agent to search the internal
    hospital directory BY NAME.

14. Do not ask the user for a hospital ID before
    attempting a name-based directory search.

15. Use the Customer Service Agent's
    search_hospital_directory tool for name-based searches.

16. Use get_hospital_info only when a hospital ID is
    available and additional directory details are needed.

17. When investigating a hospital using internal and
    external sources, compare:
    - Hospital name
    - City
    - State
    - Address, if available
    - Other identifying details returned by the tools

18. Do not assume that two similarly named hospitals
    are the same entity.

19. If multiple internal directory matches are returned,
    inspect the candidates and explain any ambiguity.

20. If internal and external information differs,
    determine whether the difference is meaningful.

21. If a meaningful discrepancy exists, delegate a
    focused follow-up task to the appropriate specialist.

22. If the internal directory returns no matching record,
    report that no matching record was found in the
    searched directory. Do not claim the hospital does
    not exist in reality.

RESEARCH AND EVIDENCE RULES:

23. Base all factual conclusions on actual specialist
    tool responses.

24. Never invent hospital names, addresses, IDs,
    claims information, policy details, or research findings.

25. Do not claim that a specialist searched a source
    unless its tool execution returned supporting results.

26. Clearly distinguish:
    - Verified internal findings
    - Verified external findings
    - Uncertain findings
    - Missing information

27. If a specialist cannot perform a requested task,
    explain the limitation instead of pretending
    the task was completed.

28. Never bypass the Customer Service Agent's existing
    authorization and verified-member safeguards.

29. Never expose another member's private information.

30. Stop when the user's objective has been sufficiently
    answered or when further investigation is not useful.

31. Provide one clear final answer explaining:
    - Tasks delegated
    - Findings from each specialist
    - Comparison of the findings
    - Any discrepancies
    - Follow-up actions taken
    - Remaining uncertainty

IMPORTANT BEHAVIOUR:

Do not simply route the request and immediately finish.

Plan, delegate, observe, evaluate, and adapt your actions
based on the information returned by the specialist agents.

Do not repeatedly call a specialist without a specific
reason.

Your goal is to coordinate the specialists and produce
a reliable answer, not to invent missing information.
"""



## 12. Create the Agentic Supervisor

We will now create the supervisor using LangChain's `create_agent`.

Unlike a normal Python function with fixed routing rules, this agent can decide which tools to call based on the user's request and the results it receives.

The agent can call:
- Research Agent tool
- Customer Service Agent tool

It can also make additional tool calls when its previous results reveal that more information is required.

The agent's execution will be bounded using a recursion limit to prevent uncontrolled loops.


In [0]:

supervisor_agent = create_agent(
    model=supervisor_llm,
    tools=supervisor_tools,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT
)

print("Agentic Supervisor created successfully.")


Agentic Supervisor created successfully.



## 13. Execute the Supervisor with a Safety Limit

An agentic supervisor can make multiple decisions and call tools repeatedly.

However, we do not want it to continue indefinitely.

We will set a recursion limit for each request.

This limits the number of graph execution steps. It is not an exact count of specialist calls, because the supervisor also needs model-execution steps.


In [0]:

MAX_RECURSION_STEPS = 20


def ask_supervisor(user_request):

    result = supervisor_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_request
                }
            ]
        },
        config={
            "recursion_limit": MAX_RECURSION_STEPS
        }
    )

    return result


print("Supervisor execution function created.")


Supervisor execution function created.


In [0]:

# STEP 2: Generate a trace for the Multi-Agent Supervisor

trace_test_question = """
Check whether Fortis Escorts Hospital Jaipur
is present in the hospital directory.
"""

trace_test_result = ask_supervisor(trace_test_question)

print("Supervisor execution completed.")





[Supervisor -> Customer Service Agent]
Task: search_hospital_directory by name Fortis Escorts Hospital Jaipur
[Customer Service Agent completed]
Supervisor execution completed.


Trace(trace_id='tr-cdfe723aff2fdea0e037adb6c09a4c99')

In [0]:

# STEP 3: Define our first MLflow evaluation scorers
# (inline definitions commented out — these judges are now registered on
#  MLflow experiment 4461081017520248 and are fetched by name below)

# from mlflow.genai.scorers import RelevanceToQuery, Guidelines

# Judge 1: Does the Supervisor answer the user's actual question?
# relevance_judge = RelevanceToQuery(
#     name="supervisor_answer_relevance"
# )

# Judge 2: Does the Supervisor follow our architecture's rules?
# supervisor_guidelines_judge = Guidelines(
#     name="supervisor_behavior_guidelines",
#     guidelines=[
#         "Use the Customer Service Agent for questions about internal "
#         "claims, members, policies, and hospital directory data.",

#         "Use the Research Agent for questions requiring publicly "
#         "available external information.",

#         "Use both specialist agents when the request genuinely "
#         "requires both internal and external information.",

#         "Clearly distinguish internal information from external "
#         "research.",

#         "Do not invent facts, tool results, or information that "
#         "was not found.",

#         "When an internal search returns no matching result, "
#         "clearly communicate that no match was found."
#     ]
# )

# print("Relevance judge:", relevance_judge.name)
# print("Guidelines judge:", supervisor_guidelines_judge.name)
# print("STEP 3 completed.")

# Scorers are now registered on the experiment — fetch by name instead of redefining
import mlflow
from mlflow.genai.scorers import get_scorer

mlflow.set_experiment(experiment_id="4461081017520248")
relevance_judge = get_scorer(name="supervisor_answer_relevance")
supervisor_guidelines_judge = get_scorer(name="supervisor_behavior_guidelines")

print("Fetched registered scorers:", relevance_judge.name, "and", supervisor_guidelines_judge.name)


Fetched registered scorers: supervisor_answer_relevance and supervisor_behavior_guidelines


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


In [0]:

# STEP 6A: Improve the custom routing judge
# (inline definition commented out — this scorer is now registered on
#  MLflow experiment 4461081017520248 and is fetched by name below)

# from mlflow.genai.judges import make_judge

# routing_judge = make_judge(
#     name="supervisor_agent_routing",
#     model="databricks",
#     feedback_value_type=bool,

#     instructions="""
#     Evaluate the routing decisions of this multi-agent supervisor.

#     User request:
#     {{ inputs }}

#     Expected agents:
#     {{ expectations }}

#     Actual execution trace:
#     {{ trace }}

#     Available specialists:

#     1. Research Agent:
#        Handles publicly available external information.

#     2. Customer Service Agent:
#        Handles internal claims, members, policies, and
#        hospital directory information.

#     Evaluation rules:

#     - Compare the actual specialist delegations in the trace
#       against the expected agents.
#     - All expected specialists must be called.
#     - Unnecessary specialist calls should be considered incorrect.
#     - Examine actual tool calls and nested agent execution.
#     - Do not mistake a tool's unsuccessful search result
#       for an incorrect routing decision.
#     - Evaluate routing, not the factual accuracy of the final answer.

#     Return a valid JSON object with exactly these keys:

# {
#   "result": true,
#   "rationale": "Brief explanation of the routing decision."
# }

# The result field must be a boolean.

# Do not include Markdown formatting.
# Do not wrap the JSON in code fences.
# Do not include any text before or after the JSON.
#     """
# )

# print("Updated routing judge:", routing_judge.name)
# print("Expected-agent comparison: ENABLED")

# Scorers are now registered on the experiment — fetch by name instead of redefining
import mlflow
from mlflow.genai.scorers import get_scorer

mlflow.set_experiment(experiment_id="4461081017520248")
routing_judge = get_scorer(name="supervisor_agent_routing")

print("Fetched registered scorer:", routing_judge.name)


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Fetched registered scorer: supervisor_agent_routing


In [0]:

# STEP 5: Create the Supervisor evaluation dataset

eval_dataset = [

    # TEST 1: Internal information
    {
        "inputs": {
            "user_request": (
                "Check whether Fortis Escorts Hospital Jaipur "
                "is present in the hospital directory."
            )
        },
        "expectations": {
            "expected_agents": ["Customer Service Agent"],
            "reason": "The request requires an internal directory lookup."
        }
    },

    # TEST 2: External research
    {
        "inputs": {
            "user_request": (
                "the publicly available background of "
                "Fortis Escorts Hospital Jaipur."
            )
        },
        "expectations": {
            "expected_agents": ["Research Agent"],
            "reason": "The request requires public external research."
        }
    },

    # TEST 3: Combined internal and external request
    {
        "inputs": {
            "user_request": (
                "Check whether Fortis Escorts Hospital Jaipur "
                "exists in our hospital directory and "
                "tell me its publicly available background."
            )
        },
        "expectations": {
            "expected_agents": [
                "Customer Service Agent",
                "Research Agent"
            ],
            "reason": (
                "The request requires both an internal lookup "
                "and external research."
            )
        }
    }
]


# Prediction function used by MLflow evaluation
def supervisor_predict(user_request: str):

    result = ask_supervisor(user_request)

    return extract_agent_response(result)


print("Evaluation dataset created.")
print("Total test cases:", len(eval_dataset))

for i, sample in enumerate(eval_dataset, start=1):
    print(f"\nTest {i}:")
    print("Question:", sample["inputs"]["user_request"])
    print(
        "Expected agents:",
        sample["expectations"]["expected_agents"]
    )

print("\nSTEP 5 completed.")


Evaluation dataset created.
Total test cases: 3

Test 1:
Question: Check whether Fortis Escorts Hospital Jaipur is present in the hospital directory.
Expected agents: ['Customer Service Agent']

Test 2:
Question: the publicly available background of Fortis Escorts Hospital Jaipur.
Expected agents: ['Research Agent']

Test 3:
Question: Check whether Fortis Escorts Hospital Jaipur exists in our hospital directory and tell me its publicly available background.
Expected agents: ['Customer Service Agent', 'Research Agent']

STEP 5 completed.


In [0]:

# STEP 6B: Run MLflow GenAI evaluation

evaluation_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=supervisor_predict,
    scorers=[
        relevance_judge,
        supervisor_guidelines_judge,
        routing_judge
    ]
)

print("Evaluation completed successfully!")
print(evaluation_results)


2026/11/26 03:14:25 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/26 03:14:25 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.



[Supervisor -> Customer Service Agent]
Task: search_hospital_directory name=Fortis Escorts Hospital Jaipur
[Customer Service Agent completed]


Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]


[Supervisor -> Research Agent]
Task: Provide publicly available background information on Fortis Escorts Hospital Jaipur.

[Supervisor -> Customer Service Agent]
Task: search_hospital_directory name='Fortis Escorts Hospital Jaipur'

[Supervisor -> Customer Service Agent]
Task: search_hospital_directory by name "Fortis Escorts Hospital Jaipur"
[Customer Service Agent completed]

[Supervisor -> Research Agent]
Task: Gather publicly available background information on Fortis Escorts Hospital Jaipur, including location, services, history, and any notable facts.
[Research Agent completed]

[Supervisor -> Customer Service Agent]
Task: search_hospital_directory name: Fortis Escorts Hospital Jaipur
[Customer Service Agent completed]

[Supervisor -> Research Agent]
Task: Search Wikipedia and other publicly available sources for background information on Fortis Escorts Hospital Jaipur, including its history, services, location, and any notable facts.
[Research Agent completed]
[Customer Service

Evaluation completed successfully!
EvaluationResult(
  run_id: ef9e8320af3e42fea8cf5175a5622618
  metrics:
    supervisor_answer_relevance/mean: 1.0
    supervisor_behavior_guidelines/mean: 1.0
    supervisor_agent_routing/mean: 0.6666666666666666
  result_df: 3 rows x 20 cols
)


## 14. Inspect the Supervisor's Execution

We need to inspect what happened during a supervisor run.

The output may contain:
- Supervisor messages.
- Tool calls made by the supervisor.
- Responses returned by specialist agents.
- The final response.

This allows us to understand how the supervisor completed the task instead of looking only at its final answer.


In [0]:

def display_supervisor_execution(result):

    messages = result.get("messages", [])

    for index, message in enumerate(messages, start=1):

        print("\n" + "=" * 60)
        print("MESSAGE", index)
        print("=" * 60)

        print("Message type:", getattr(message, "type", "unknown"))

        tool_calls = getattr(message, "tool_calls", [])

        if tool_calls:
            print("\nTool calls:")
            for call in tool_calls:
                print("Tool:", call.get("name"))
                print("Arguments:", call.get("args"))

        content = getattr(message, "content", "")

        if content:
            print("\nContent:")
            print(content)

    return messages


print("Execution inspection function created.")


Execution inspection function created.


## 15. Test 1 — Internal Customer Service Request

### User Request

"Explain what information is available for claim CLM-10341."

### Expected Behaviour

The supervisor should recognize that this is an internal claims question.

It should delegate the task to the Customer Service Agent rather than unnecessarily calling the Research Agent.

This tests whether the supervisor can select the appropriate specialist.


In [0]:

request_1 = """
retrieve the available information for claim CLM-10341.

Explain the claim's current status and other relevant details
returned by the internal system.

Do not invent any missing information.
"""

result_1 = ask_supervisor(request_1)

display_supervisor_execution(result_1)



[Supervisor -> Customer Service Agent]
Task: Retrieve claim information for claim CLM-10341
[Customer Service Agent completed]

MESSAGE 1
Message type: human

Content:

retrieve the available information for claim CLM-10341.

Explain the claim's current status and other relevant details
returned by the internal system.

Do not invent any missing information.


MESSAGE 2
Message type: ai

Tool calls:
Tool: delegate_to_customer_service_agent
Arguments: {'task': 'Retrieve claim information for claim CLM-10341'}

MESSAGE 3
Message type: tool

Content:
**Claim CLM-10341 – Mumbai Care Hospital 11**

| Item | Detail |
|------|--------|
| **Status** | Submitted |
| **Admission date** | 2026‑03‑11 |
| **Discharge date** | 2026‑03‑19 |
| **Billed amount** | ₹132,605.57 |
| **Approved amount** | Not yet approved (null) |
| **Policy** | POL‑0317 (Plan: CORP_GROUP, Sum insured: ₹500,000) |
| **Room rent cap** | ₹5,000 per day |
| **Hospital** | Mumbai Care Hospital 11, Mumbai |

The claim is curre

[HumanMessage(content="\nretrieve the available information for claim CLM-10341.\n\nExplain the claim's current status and other relevant details\nreturned by the internal system.\n\nDo not invent any missing information.\n", additional_kwargs={}, response_metadata={}, id='f34fe566-e1a4-49eb-9d28-d87a0e868d4d'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_656be035-e849-44f7-a18b-feef3f1e92c4', 'function': {'arguments': '{"task":"Retrieve claim information for claim CLM-10341"}', 'name': 'delegate_to_customer_service_agent'}, 'type': 'function'}]}, response_metadata={'usage': {'prompt_tokens': 1066, 'completion_tokens': 72, 'total_tokens': 1138}, 'prompt_tokens': 1066, 'completion_tokens': 72, 'total_tokens': 1138, 'model': 'gpt-oss-20b-080525', 'model_name': 'gpt-oss-20b-080525', 'finish_reason': 'tool_calls'}, id='lc_run--01a217db-0c95-748a-8a9d-40acc5a55382-0', tool_calls=[{'name': 'delegate_to_customer_service_agent', 'args': {'task': 'Retrieve claim inform

Trace(trace_id='tr-ef61c8a8a7922985b2bc2fdf987f5ecf')

## 16. Test 2 — External Research Request

### User Request

"Research the publicly available background of Fortis Escorts Hospital Jaipur."

### Expected Behaviour

The supervisor should recognize that this requires external information.

It should delegate the task to the External Tools Research Agent.

The Customer Service Agent should not be called unless the user also asks for an internal directory or claims-data check.


In [0]:

request_2 = """
Tell me about the publicly available background of Fortis Escorts Hospital.

Summarize the findings and mention any limitations
in the available information.
"""

result_2 = ask_supervisor(request_2)

display_supervisor_execution(result_2)



[Supervisor -> Research Agent]
Task: Gather publicly available background information on Fortis Escorts Hospital, including history, services, location, notable facts, and any other relevant public details. Summarize findings and note limitations.
[Research Agent completed]

[Supervisor -> Customer Service Agent]
Task: search_hospital_directory name: "Fortis Escorts Hospital"
[Customer Service Agent completed]

MESSAGE 1
Message type: human

Content:

Tell me about the publicly available background of Fortis Escorts Hospital.

Summarize the findings and mention any limitations
in the available information.


MESSAGE 2
Message type: ai

Tool calls:
Tool: delegate_to_research_agent
Arguments: {'task': 'Gather publicly available background information on Fortis Escorts Hospital, including history, services, location, notable facts, and any other relevant public details. Summarize findings and note limitations.'}

MESSAGE 3
Message type: tool

Content:
Background:
Fortis Healthcare Limite

[HumanMessage(content='\nTell me about the publicly available background of Fortis Escorts Hospital.\n\nSummarize the findings and mention any limitations\nin the available information.\n', additional_kwargs={}, response_metadata={}, id='b4a683bc-012d-4c80-b402-ce1cab6fa717'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_962e5ca1-3bd4-4627-b311-cf054a3da5d5', 'function': {'arguments': '{"task":"Gather publicly available background information on Fortis Escorts Hospital, including history, services, location, notable facts, and any other relevant public details. Summarize findings and note limitations."}', 'name': 'delegate_to_research_agent'}, 'type': 'function'}]}, response_metadata={'usage': {'prompt_tokens': 1058, 'completion_tokens': 85, 'total_tokens': 1143}, 'prompt_tokens': 1058, 'completion_tokens': 85, 'total_tokens': 1143, 'model': 'gpt-oss-20b-080525', 'model_name': 'gpt-oss-20b-080525', 'finish_reason': 'tool_calls'}, id='lc_run--01a2167d-c3e3-76af-

Trace(trace_id='tr-a4e3f730bf60451d38486623fb6a6934')


## 17. Test 3 — Multi-Agent Collaboration

This is the most important demonstration of our project.

### User Request

"Check whether Fortis Escorts Hospital Jaipur exists in our internal hospital directory. Also research the hospital externally and compare the information."

### What Makes This Test Different?

This request requires two different sources of information.

The supervisor should:

1. Understand that the task needs both internal and external information.
2. Delegate the directory lookup to the Customer Service Agent.
3. Delegate the external research to the Research Agent.
4. Inspect both responses.
5. Compare the hospital name and address.
6. Identify missing or conflicting information.
7. Decide whether another lookup is needed.
8. Produce a combined report.

The supervisor may perform these tasks sequentially. It does not have to call both agents simultaneously.

**Important:** The hospital name is an example. We must not assume that a matching record exists in our internal directory.


In [0]:

request_3 = """
Investigate whether Fortis Escorts Hospital Jaipur exists
in our internal hospital directory.

Also research the hospital.

Your plan should include:

1. Ask the Customer Service Agent to search the internal
   hospital directory.

2. Ask the Research Agent to investigate the hospital
   using publicly available information.

3. Compare the returned hospital names and addresses.

4. Check whether the information is consistent.

5. If there is a meaningful discrepancy or missing detail,
   decide whether a focused follow-up task is required.

6. Do not assume two similarly named hospitals are the
   same entity.

7. Do not invent information.

Provide a final report that separates verified findings
from uncertain findings.
"""

result_3 = ask_supervisor(request_3)

display_supervisor_execution(result_3)



[Supervisor -> Customer Service Agent]
Task: search hospital directory for Fortis Escorts Hospital Jaipur
[Customer Service Agent completed]

[Supervisor -> Research Agent]
Task: Investigate Fortis Escorts Hospital Jaipur using publicly available information. Provide name, city, state, address, and any other identifying details.


Tool 'duckduckgo_search' failed on query 'Fortis Escorts Hospital Jaipur address': No results found.
Tool 'wikipedia' failed on query 'Fortis Escorts Hospital Jaipur': Expecting value: line 1 column 1 (char 0)


[Research Agent completed]

MESSAGE 1
Message type: human

Content:

Investigate whether Fortis Escorts Hospital Jaipur exists
in our internal hospital directory.

Also research the hospital.

Your plan should include:

1. Ask the Customer Service Agent to search the internal
   hospital directory.

2. Ask the Research Agent to investigate the hospital
   using publicly available information.

3. Compare the returned hospital names and addresses.

4. Check whether the information is consistent.

5. If there is a meaningful discrepancy or missing detail,
   decide whether a focused follow-up task is required.

6. Do not assume two similarly named hospitals are the
   same entity.

7. Do not invent information.

Provide a final report that separates verified findings
from uncertain findings.


MESSAGE 2
Message type: ai

Tool calls:
Tool: delegate_to_customer_service_agent
Arguments: {'task': 'search hospital directory for Fortis Escorts Hospital Jaipur'}

MESSAGE 3
Message type: tool

C

[HumanMessage(content='\nInvestigate whether Fortis Escorts Hospital Jaipur exists\nin our internal hospital directory.\n\nAlso research the hospital.\n\nYour plan should include:\n\n1. Ask the Customer Service Agent to search the internal\n   hospital directory.\n\n2. Ask the Research Agent to investigate the hospital\n   using publicly available information.\n\n3. Compare the returned hospital names and addresses.\n\n4. Check whether the information is consistent.\n\n5. If there is a meaningful discrepancy or missing detail,\n   decide whether a focused follow-up task is required.\n\n6. Do not assume two similarly named hospitals are the\n   same entity.\n\n7. Do not invent information.\n\nProvide a final report that separates verified findings\nfrom uncertain findings.\n', additional_kwargs={}, response_metadata={}, id='ed25efda-c8af-495c-adef-7a60f97e53cc'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_47bd49d0-ba73-4841-a8d9-ee1e3eff4642', 'function': {'ar

Trace(trace_id='tr-149f507c97c03b2110f0fb4469bbbd79')


## 18. Analyze Specialist Delegation

We now want to inspect how many times the supervisor called each specialist.

This helps us understand whether the supervisor:
- Called only one specialist.
- Called both specialists.
- Called a specialist again for follow-up work.

Remember that more tool calls do not automatically mean better performance. We need to inspect whether each call was useful.


In [0]:

def inspect_tool_calls(result):

    messages = result.get("messages", [])

    tool_calls = []

    for message in messages:

        calls = getattr(message, "tool_calls", [])

        for call in calls:

            tool_calls.append({
                "tool_name": call.get("name"),
                "arguments": call.get("args")
            })

    research_count = sum(
        1 for call in tool_calls
        if call["tool_name"] == "delegate_to_research_agent"
    )

    customer_count = sum(
        1 for call in tool_calls
        if call["tool_name"] == "delegate_to_customer_service_agent"
    )

    print("Research Agent calls:", research_count)
    print("Customer Service Agent calls:", customer_count)
    print("Total specialist calls:", len(tool_calls))

    print("\nDetailed delegation:")
    for index, call in enumerate(tool_calls, start=1):
        print(f"\nCall {index}")
        print("Agent:", call["tool_name"])
        print("Task:", call["arguments"])

    return tool_calls


multi_agent_calls = inspect_tool_calls(result_3)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6887829546912640>, line 42
     37         print("Task:", call["arguments"])
     39     return tool_calls
---> 42 multi_agent_calls = inspect_tool_calls(result_3)

NameError: name 'result_3' is not defined

## 19. Evaluate Agentic Behaviour

A supervisor is not agentic merely because it can call two agents.

We should check whether it demonstrates the following behaviours:

| Behaviour | What to look for |
|---|---|
| Planning | Identifies multiple subtasks |
| Delegation | Selects appropriate specialists |
| Observation | Reads specialist responses |
| Evaluation | Checks whether results are sufficient |
| Adaptation | Requests follow-up work when needed |
| Completion | Stops and produces a final answer |

### Important Evaluation Point

If the supervisor calls both agents once and combines their responses, it demonstrates multi-agent delegation.

If it also evaluates the results and decides to request additional work when a genuine information gap exists, it demonstrates adaptive agentic behaviour.

We should test both situations rather than assume that the system will always behave correctly.


## 20. Test 4 — Follow-up Decision

In this test, we provide the supervisor with the findings from Test 3.

The supervisor must:

1. Review the previous internal and external findings.
2. Identify whether any important information is missing.
3. Decide whether a follow-up investigation is necessary.
4. If necessary, delegate a focused task to the appropriate specialist.
5. Avoid repeating the entire investigation unnecessarily.
6. Produce an updated conclusion.

If the previous results are already sufficient, the supervisor should explain why no further delegation is needed.

This tests whether the supervisor can make an adaptive follow-up decision using the results of an earlier execution.

In [0]:
# ---------------------------------------------------------
# TEST 4 — FOLLOW-UP DECISION
# ---------------------------------------------------------

# Extract the findings from Test 3.
previous_findings = extract_agent_response(result_3)

print("Previous findings:")
print(previous_findings)


# Ask the supervisor to evaluate the previous findings.
request_4 = f"""
You previously investigated Fortis Escorts Hospital Jaipur
using the internal hospital directory and external research.

Here are the findings from that investigation:

{previous_findings}

Now evaluate these findings.

Your responsibilities:

1. Identify any missing or inconsistent information.

2. Decide whether another investigation is necessary.

3. If more information is required, delegate a focused
   follow-up task to the appropriate specialist.

4. Do not repeat the entire investigation unnecessarily.

5. If the available information is sufficient, do not
   make unnecessary specialist calls.

6. Provide an updated conclusion explaining your decision.
"""

result_4 = ask_supervisor(request_4)

display_supervisor_execution(result_4)

Previous findings:
**Final Report – Investigation of “Fortis Escorts Hospital Jaipur”**

| Step | Action | Outcome |
|------|--------|---------|
| 1 | **Internal Directory Search** (Customer Service Agent) | No record of a hospital named “Fortis Escorts Hospital Jaipur” was returned from the internal directory. |
| 2 | **External Research** (Research Agent) | The agent confirmed that a hospital called “Fortis Escorts Hospital” exists in Jaipur, Rajasthan, India. No address or additional identifying details were retrieved from publicly‑available sources. |
| 3 | **Comparison** | • **Name** – Both sources agree on the hospital name (Fortis Escorts Hospital). <br>• **City/State** – Both sources agree on Jaipur, Rajasthan. <br>• **Address** – Internal directory: *not available* (no record). External research: *not available* (tool could not fetch). |
| 4 | **Consistency Check** | The information is partially consistent: name and location match. However, the internal directory lacks any rec

[HumanMessage(content='\nYou previously investigated Fortis Escorts Hospital Jaipur\nusing the internal hospital directory and external research.\n\nHere are the findings from that investigation:\n\n**Final Report – Investigation of “Fortis Escorts Hospital Jaipur”**\n\n| Step | Action | Outcome |\n|------|--------|---------|\n| 1 | **Internal Directory Search** (Customer Service Agent) | No record of a hospital named “Fortis Escorts Hospital Jaipur” was returned from the internal directory. |\n| 2 | **External Research** (Research Agent) | The agent confirmed that a hospital called “Fortis Escorts Hospital” exists in Jaipur, Rajasthan, India. No address or additional identifying details were retrieved from publicly‑available sources. |\n| 3 | **Comparison** | • **Name** – Both sources agree on the hospital name (Fortis Escorts Hospital). <br>• **City/State** – Both sources agree on Jaipur, Rajasthan. <br>• **Address** – Internal directory: *not available* (no record). External researc

Trace(trace_id='tr-6ae26a79dba1c083246a5b1619e19f8e')

In [0]:
for i, row in evaluation_results.result_df.iterrows():

    print(f"\n{'='*60}")
    print(f"TEST {i + 1}")
    print(f"{'='*60}")

    print("\nRouting Judge Error:")
    print(row.get("supervisor_agent_routing/error_message", "n/a"))

    print("\nRelevance Score:")
    print(row.get("supervisor_answer_relevance/value", "n/a"))

    print("\nGuidelines Score:")
    print(row.get("supervisor_behavior_guidelines/value", "n/a"))


TEST 1

Routing Judge Error:
n/a

Relevance Score:
yes

Guidelines Score:
yes

TEST 2

Routing Judge Error:
n/a

Relevance Score:
None

Guidelines Score:
yes

TEST 3

Routing Judge Error:
n/a

Relevance Score:
yes

Guidelines Score:
yes


## 21. Run Your Own Query Through the Supervisor

Type any question into the `supervisor_query` input box created below.

The supervisor decides on its own whether to:

- Answer directly.
- Delegate to the Customer Service Agent (internal claims, members, policies, hospital directory).
- Delegate to the External Tools Research Agent (publicly available information).
- Combine both specialists when the query needs internal and external data.

After typing the query, run the next cell to send it through the supervisor.


In [0]:
# ---------------------------------------------------------
# INPUT BOX FOR THE QUERY
# ---------------------------------------------------------
# Creates the 'supervisor_query' input box.
# Type your query, then run the next cell.
# If you change the query later, just re-run the next cell.

dbutils.widgets.text(
    name="supervisor_query",
    defaultValue="Check the status of claim CLM-10341 and summarize what the internal system shows about it.",
    label="Query for the supervisor"
)

print("Input box created. Type your query, then run the next cell.")


Input box created. Type your query, then run the next cell.


In [0]:
# ---------------------------------------------------------
# RUN THE QUERY THROUGH THE SUPERVISOR
# ---------------------------------------------------------
# Reads the query from the input box and sends it through
# the supervisor, then prints the full execution trace.

user_query = dbutils.widgets.get("supervisor_query")

if not user_query or not user_query.strip():

    print("Please type a query in the 'supervisor_query' input box, then run this cell again.")

else:

    user_query = user_query.strip()

    print("Running query through the supervisor:\n")
    print(user_query)
    print("\n" + "=" * 60 + "\n")

    result_custom = ask_supervisor(user_query)

    display_supervisor_execution(result_custom)

    print("\n\nSpecialist delegation for this query:\n")


Running query through the supervisor:

I am the admin of the insurance company I want the member detail for



MESSAGE 1
Message type: human

Content:
I am the admin of the insurance company I want the member detail for

MESSAGE 2
Message type: ai

Content:
I’m happy to help retrieve the member’s details. Could you please provide the member’s ID (or full name if you prefer) so I can look up the correct record in our system?


Specialist delegation for this query:



Trace(trace_id='tr-40a0c20cec59307b05014566f19eef66')